# 01 · SPHEREx data walkthrough (Phase 0)

Goal: understand the data **before** building, and answer the questions in `docs/PLAN.md` §2.
Write each answer into `docs/DATA.md` as we go.

Run from the repo root: `.venv\Scripts\jupyter lab`, then open this notebook.
Already verified on 14.11: the IRSA TAP query works (~18 s), Sesame resolves all five targets,
and SIMBAD's TAP endpoint returned 404 (so we use Sesame for proper motions for now).

In [ ]:
import time, re, requests
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
import astropy.units as u
from astroquery.ipac.irsa import Irsa

CACHE = Path('../cache'); CACHE.mkdir(exist_ok=True)   # git-ignored

## 1 · Targets and proper motion (from CDS Sesame)

In [ ]:
TARGETS = {
    'barnard':  "Barnard's Star",
    'epsindB':  'eps Ind B',
    'wise0855': 'WISE J085510.83-071442.5',
    'rwaur':    'RW Aur',
    'rdra':     'R Dra',
}

def sesame(name):
    """J2000 position + proper motion (mas/yr) from CDS Sesame."""
    url = 'https://cds.unistra.fr/cgi-bin/nph-sesame/-oxp/S?' + requests.utils.quote(name)
    x = requests.get(url, timeout=30).text
    def g(tag):
        m = re.search(f'<{tag}>([^<]+)</{tag}>', x)
        return float(m.group(1)) if m else None
    return dict(ra=g('jradeg'), dec=g('jdedeg'), pmra=g('pmRA') or 0.0, pmdec=g('pmDE') or 0.0)

info = {k: sesame(n) for k, n in TARGETS.items()}
for k, v in info.items():
    print(f"{k:9s} ra={v['ra']:10.5f} dec={v['dec']:+9.5f}  pm=({v['pmra']:8.1f}, {v['pmdec']:8.1f}) mas/yr")

In [ ]:
def position_at(v, t):
    """Apply proper motion from J2000 to time t (linear; fine at the 6.2-arcsec pixel scale)."""
    c = SkyCoord(v['ra']*u.deg, v['dec']*u.deg, pm_ra_cosdec=v['pmra']*u.mas/u.yr,
                 pm_dec=v['pmdec']*u.mas/u.yr, obstime=Time('J2000'), frame='icrs')
    return c.apply_space_motion(new_obstime=t)

# Q: how far has each target moved since J2000, and how much per 6 months, in SPHEREx pixels (~6.2")?
for k, v in info.items():
    total = np.hypot(v['pmra'], v['pmdec']) / 1000     # arcsec/yr
    print(f"{k:9s} {total:6.2f} arcsec/yr  ->  {total*0.5/6.2:4.2f} px per 6 months, {total*26/60:5.2f} arcmin since J2000")

## 2 · How many exposures? (per detector, and over what time span)

In [ ]:
def count_exposures(ra, dec):
    q = f"""
    SELECT p.energy_bandpassname AS band, COUNT(*) AS n,
           MIN(p.time_bounds_lower) AS tmin, MAX(p.time_bounds_lower) AS tmax
    FROM spherex.artifact a JOIN spherex.plane p ON a.planeid = p.planeid
    WHERE 1 = CONTAINS(POINT('ICRS', {ra}, {dec}), p.poly)
    GROUP BY p.energy_bandpassname ORDER BY p.energy_bandpassname"""
    return Irsa.query_tap(q).to_table()

t0 = time.time(); tab = count_exposures(info['rdra']['ra'], info['rdra']['dec'])
print(f'{time.time()-t0:.1f}s')
tab
# Seen on 14.11 for R Dra: D1/D4 ~584, D2/D5 89, D3/D6 ~574, MJD 60790-61310 (May 2025 - Oct 2026).
# Q: why do detector pairs have identical counts? Are there duplicates from QR2 + QR3 + DR1?

## 3 · Which columns exist? (look for release / version, wavelength, quality)
Look at the schema before guessing. We need to keep only DR1 + QR3.

In [ ]:
cols = Irsa.query_tap("""SELECT table_name, column_name, datatype, description
FROM TAP_SCHEMA.columns WHERE table_name IN ('spherex.plane','spherex.artifact')""").to_table()
cols.pprint(max_lines=-1, max_width=200)

## 4 · List the exposures of one detector, with cutout URLs

In [ ]:
def list_exposures(ra, dec, band='SPHEREx-D2', size_deg=0.05):
    q = f"""
    SELECT 'https://irsa.ipac.caltech.edu/' || a.uri || '?center={ra},{dec}d&size={size_deg}' AS uri,
           p.time_bounds_lower AS mjd, p.energy_bandpassname AS band
    FROM spherex.artifact a JOIN spherex.plane p ON a.planeid = p.planeid
    WHERE 1 = CONTAINS(POINT('ICRS', {ra}, {dec}), p.poly) AND p.energy_bandpassname = '{band}'
    ORDER BY p.time_bounds_lower"""
    return Irsa.query_tap(q).to_table()

exp = list_exposures(info['rdra']['ra'], info['rdra']['dec'])
print(len(exp))
exp[:5]

In [ ]:
# Q: do the exposures cluster into visits ("epochs")? Look at the gaps between them.
mjd = np.asarray(exp['mjd'])
plt.figure(figsize=(10, 2)); plt.plot(mjd, np.zeros_like(mjd), '|', ms=20); plt.xlabel('MJD'); plt.yticks([])
gaps = np.diff(mjd); print('gaps > 5 days:', np.round(gaps[gaps > 5], 1))

## 5 · Download one cutout and look at every HDU

In [ ]:
def get_cutout(url, name):
    f = CACHE / name
    if not f.exists():
        t = time.time(); f.write_bytes(requests.get(url, timeout=120).content)
        print(f'downloaded {f.stat().st_size/1024:.0f} KB in {time.time()-t:.1f}s')
    return fits.open(f)

hdul = get_cutout(exp['uri'][0], 'rdra_D2_0.fits')
hdul.info()
# Q: units of IMAGE (BUNIT)? Pixel scale? Which FLAGS bits are set?

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, name in zip(ax, ['IMAGE', 'FLAGS', 'VARIANCE', 'ZODI']):
    d = hdul[name].data
    a.imshow(d, origin='lower', cmap='gray', vmin=np.nanpercentile(d, 1), vmax=np.nanpercentile(d, 99))
    a.set_title(name)
print(repr(hdul['IMAGE'].header)[:3000])

## 6 · The wavelength trap
Each pixel sees a different wavelength (linear variable filter). What wavelength is our target seen at,
and how much does it change from one exposure to the next?

In [ ]:
def wavelength_at_target(hdul, ra, dec):
    img = hdul['IMAGE'].header
    w = WCS(img); x, y = w.world_to_pixel(SkyCoord(ra*u.deg, dec*u.deg))
    ww = WCS(img, fobj=hdul, key='W'); ww.sip = None
    lam, dlam = ww.pixel_to_world(x, y)   # per the IRSA tutorial
    return float(x), float(y), lam, dlam

print(wavelength_at_target(hdul, info['rdra']['ra'], info['rdra']['dec']))
# Next: do this for ~20 exposures and plot wavelength vs time. Same detector: how much spread?

## 7 · Blink two exposures by hand (the whole product, done crudely)
Q: are they rotated relative to each other? If so we must reproject onto a common grid (`reproject`).

In [ ]:
h2 = get_cutout(exp['uri'][len(exp)//2], 'rdra_D2_mid.fits')
for h in (hdul, h2):
    w = WCS(h['IMAGE'].header).wcs
    print(w.cd if w.has_cd() else w.pc)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for a, h in zip(ax, (hdul, h2)):
    d = h['IMAGE'].data - h['ZODI'].data
    a.imshow(d, origin='lower', cmap='gray', vmin=np.nanpercentile(d, 5), vmax=np.nanpercentile(d, 99.5))

## 8 · Still to check (write into `docs/DATA.md`)
- [ ] Cutout download time and size; how many parallel downloads are OK
- [ ] Barnard's Star: blink two epochs about 6-12 months apart; does it move ~1-2 px?
- [ ] Can a browser call IRSA directly (CORS)? Is there a SPHEREx HiPS for Aladin Lite?
- [ ] Gaia DR3 overlay: Aladin Lite built-in catalog, or a query of our own?
- [ ] Agree `docs/BUNDLE_FORMAT.md`